In [ ]:
import requests
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from sqlalchemy import create_engine
import time
import cloudscraper
import re

In [ ]:
list_am_main = 'https://www.list.am'
list_am_url = f"{list_am_main}/category/1472?n=0&bid=49&mid=957%2C3284&_a2_1=2012&_a2_2=2017&_a27=0&_a22=0&_a102=0"

In [ ]:
def get_html_response(url):
    scraper = cloudscraper.create_scraper()
    response = scraper.get(url)
    response_text = response.text
    soup = BeautifulSoup(response_text, "html.parser")
    return soup


def get_cars(soup):
    search_results = soup.find('div' ,attrs={'class':'gl'})
    return search_results

def get_hrefs(cars):
    hrefs = [list_am_main + link['href'] for link in cars.find_all('a',attrs={'class':"class"})]
    return hrefs

def get_specific_car_content(href):
    scraper = cloudscraper.create_scraper()
    html = scraper.get(href).text
    #html = requests.get(href).text
    soup = BeautifulSoup(html, "html.parser")
    return soup

def get_car_name(car_soup):
    return car_soup.find('title').text.split(' - ')[0]

def get_car_vin(car_soup):
    try:
        return car_soup.find('div',attrs={'class':"pad-left-6"}).text.strip()
    except:
        return

def get_car_year(car_soup):
    return car_soup.find('a',attrs={'class':"grey-text"}).text

def get_car_metadata(car_soup):
    data_dict = {}
    all_details = car_soup.find('div', class_='vi')
    all_details

    car_detils = all_details.find_all('div',class_='attr g new')
    for detail in car_detils:
        sub_details = detail.find_all('div',class_='at2')
        for sub_detail in sub_details:
            names = sub_detail.find_all('div')
            for name in names[1:]:
                titles = name.find_all('p')
                if len(titles) > 1:
                    tag_element,name_element = titles
                else:
                    tag_element =  titles[0]
                    name_element = titles[0]
                data_dict[name_element.text] = tag_element.text
                    
    return data_dict


def get_add_info(car_soup):
    add_infos = car_soup.find_all('div',attrs={'class':"nottii bltitle medium"})
    count = 0
    add_exists = False
    for i in add_infos:
        if i.text == 'Լրացուցիչ':
            add_exists = True
            break
        else:
            count += 1
    if add_exists:
        add_info = car_soup.find_all('div',attrs={'class':"ad-options"})[count].text.strip()
    else:
        add_info = None
    return add_info

def get_car_seller_phone(car_soup):
    try:
        return car_soup.find('a' , attrs={'id':"callBtnOptional1"}).text.strip()
    except:
        return car_soup.find('a' , attrs={'id':"callBtn1"}).text.strip()

def get_seller_id(car_soup):
    return car_soup.find('a',class_ = 'n')['href'].split('/')[-1]

def get_car_price(car_soup):
    return car_soup.find('span', class_='price x').text.strip()

def get_add_info(car_soup):
    def extract_post_id(span):
        return span.text.split(' ')[-1]
        return re.search(r'\d+', span.text).group()

    def extract_create_date(span):
        return span.text.split(' ')[-1]

    def extract_update_date(span):
        if span: 
            return ' '.join(span.text.split(' ')[-2:]) 
    
    description = car_soup.find('div', class_='vi').find('div', class_='body').text
    other_info = car_soup.find('div', class_='vi').find('div', class_='footer').find_all('span')
    
    if len(other_info) == 3:
        post_id_span,create_date_span,update_date_span = other_info
    else:
        post_id_span,create_date_span = other_info
        update_date_span = None
    post_id,create_date,update_date = extract_post_id(post_id_span) ,extract_create_date(create_date_span),extract_update_date(update_date_span)
    return description,post_id,create_date,update_date

In [ ]:
soup = get_html_response(list_am_url)
cars = get_cars(soup)
hrefs = get_hrefs(cars)

In [ ]:
print(f'Found {len(hrefs)} cars')

Found 26 cars


In [ ]:
final_data = {
    'car_name': [],
    'year': [],
    'add_info': [],
    'phone': [],
    'price': [],
    'Վազքը': [],
    'Թափքը': [],
    'Շարժիչը': [],
    'Փոխանցման տուփը': [],
    'Ղեկը': [],
    'Գույնը': [],
    'Վիճակը': [],
    'Սրահի գույնը': [],
    'Շարժիչի ծավալը': [],
    'Դռների քանակը': [],
    'Ձիաուժը': [],
    'Մոդիֆիկացիան': [],
    'Մխոցների քանակը': [],
    'Քարշակը': [],
    'Անվահեծերը': [],
    'Link':[]}
count = 0
cars_list = []
for href in hrefs:
    car_soup = get_specific_car_content(href)
    car_name = get_car_name(car_soup)
    price = get_car_price(car_soup)
    metadata = get_car_metadata(car_soup=car_soup)
    description,post_id,create_date,update_date = get_add_info(car_soup)
    seller_id = get_seller_id(car_soup)
    metadata['car_name']=car_name
    metadata['price'] = price
    metadata['description'] = description
    metadata['post_id'] = post_id
    metadata['create_date'] = create_date
    metadata['update_date'] = update_date
    metadata['Link'] = href
    cars_list.append(metadata)

https://www.list.am/item/23820362?ld_src=2
[<span>Հայտարարության համարը 23820362</span>, <span content="2026-05-29T14:35:10+00:00" itemprop="datePosted">Տեղադրված է 29.05.2026</span>, <span>Թարմացվել է 06.08.2026, 11:44</span>]
https://www.list.am/item/23236278?ld_src=2
[<span>Հայտարարության համարը 23236278</span>, <span content="2025-12-17T07:32:04+00:00" itemprop="datePosted">Տեղադրված է 17.12.2025</span>, <span>Թարմացվել է 05.08.2026, 18:18</span>]
https://www.list.am/item/23995786?ld_src=2
[<span>Հայտարարության համարը 23995786</span>, <span content="2026-07-16T21:50:01+00:00" itemprop="datePosted">Տեղադրված է 17.07.2026</span>, <span>Թարմացվել է 05.08.2026, 15:17</span>]
https://www.list.am/item/23006282?ld_src=2
[<span>Հայտարարության համարը 23006282</span>, <span content="2025-10-15T15:31:20+00:00" itemprop="datePosted">Տեղադրված է 15.10.2025</span>, <span>Թարմացվել է 05.08.2026, 15:09</span>]
https://www.list.am/item/23587943?ld_src=2
[<span>Հայտարարության համարը 23587943</span>,

In [20]:
df = pd.DataFrame(cars_list)

In [21]:
df['scrape_date'] = pd.to_datetime(datetime.today())

In [102]:


# 2. Define your PostgreSQL credentials
db_user = 'postgres'
db_password = 'postgres'
db_host = 'localhost'       # Use your server IP if it is not hosted locally
db_port = '5432'            # 5432 is the default PostgreSQL port
db_name = 'car_db'

# 3. Create the SQLAlchemy engine for PostgreSQL
connection_string = f'postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}'
engine = create_engine(connection_string)


In [103]:

# 4. Append the data to the table
df.to_sql(
    name='auto_am_listings', 
    con=engine, 
    if_exists='append', 
    index=False
)

print(f"Successfully appended {len(df)} rows to PostgreSQL for {df['scrape_date'].iloc[0]}")

Successfully appended 3 rows to PostgreSQL for 2026-08-02 13:10:56.164582
